# Experiment B — corrected macro-F1 and true hyperparameters (standalone)

Fixes two issues raised by *Algorithms* reviewer #4:

1. **Comment 5**: the pipeline's `precision_score`/`recall_score`/`f1_score` calls for
   Experiment B never pass `average=`, so scikit-learn defaults to `average='binary'`
   (positive-class-only metrics). The manuscript labels this "F1-macro", which it is not.
   This notebook reruns Experiment B with `average='macro'` to get the real macro-F1.
2. **Comment 7**: the original hyperparameter search (`PROCESO_MODELADO.ipynb`,
   `RandomizedSearchCV` with `cv=PredefinedSplit(...)`, scoring='accuracy') found
   `criterion='entropy'` as the best value for Random Forest in Experiment B — not
   `'gini'`, which is what the current pipeline (and the manuscript) actually uses.
   This notebook reruns with the value the search actually found.

Both changes are applied together, k=10 stratified CV, RandomOverSampler inside-fold
(unchanged from the existing protocol). Runtimes for Experiment B are fast
(~2 min for RF, ~1 min for GB per the paper's own Table 13), so this should complete
in well under 10 minutes total.

Adjust `DRIVE_PARQUET` in the setup cell before running.


In [4]:
# ── COLAB SETUP (sin Google Drive) ──────────────────────────────────────
from google.colab import files
import os

LOCAL_PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'

if not os.path.exists(LOCAL_PARQUET):
    print('Sube el archivo parquet cuando aparezca el botón...')
    uploaded = files.upload()
    subido = list(uploaded.keys())[0]
    if subido != os.path.basename(LOCAL_PARQUET):
        os.rename(subido, LOCAL_PARQUET)
    print(f'Parquet listo en {LOCAL_PARQUET}')
else:
    print(f'Parquet ya disponible en {LOCAL_PARQUET}')

!pip install imbalanced-learn -q
print('Setup complete.')

Sube el archivo parquet cuando aparezca el botón...


Saving part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet to part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet
Parquet listo en /content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet
Setup complete.


In [5]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import RandomOverSampler

df_raw = pd.read_parquet(LOCAL_PARQUET)

for col in df_raw.select_dtypes(include='string').columns:
    df_raw[col] = df_raw[col].astype(object)
for col in ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME', 'YEAR']:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(int)

df_raw['GROUP_CPV'] = df_raw['CPV'].astype(str).str[:2]
cat_cols = ['B_MULTIPLE_CAE', 'B_ON_BEHALF', 'CAE_TYPE', 'MAIN_ACTIVITY',
            'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT', 'GROUP_CPV']
for col in cat_cols:
    df_raw[col] = df_raw[col].astype(str)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_arr = ohe.fit_transform(df_raw[cat_cols])
ohe_cols = ohe.get_feature_names_out(cat_cols)

num_cols = ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS']
df_proc = pd.concat([
    df_raw[num_cols].reset_index(drop=True),
    pd.DataFrame(ohe_arr, columns=ohe_cols)
], axis=1)

df_raw['SME_WIN'] = df_raw['B_CONTRACTOR_SME'].astype(str)\
                      .str.contains('y', case=False, na=False).astype(int)
df_proc['SME_WIN'] = df_raw['SME_WIN'].reset_index(drop=True)

used_features_B2 = [
    "B_MULTIPLE_CAE_n", "B_ON_BEHALF_n",
    "GROUP_CPV_45", "GROUP_CPV_33", "GROUP_CPV_15",
    "TYPE_OF_CONTRACT_w", "MAIN_ACTIVITY_health", "ISO_COUNTRY_CODE_si",
    "NUMBER_AWARDS", "LOTS_NUMBER", "NUMBER_OFFERS",
    "CAE_TYPE_3", "MAIN_ACTIVITY_general public\\services",
    "CAE_TYPE_4", "CAE_TYPE_5", "ISO_COUNTRY_CODE_lu"
]
X_B2 = df_proc[used_features_B2].astype(float).to_numpy()
y_B2 = df_proc['SME_WIN'].to_numpy()

print(f'X_B2 shape: {X_B2.shape}')


X_B2 shape: (159752, 16)


In [6]:
def validacion_cruzada_binaria_macro(X, y, modelo, nombre_modelo, k=10):
    """
    Same protocol as the paper's validacion_cruzada_binaria, but with
    average='macro' passed explicitly to precision/recall/F1 (Comment 5),
    so the reported numbers are true macro-F1, not positive-class-only F1.
    Also reports the positive-class-only values alongside, for direct
    comparison against the currently published (mislabeled) figures.
    """
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    ros = RandomOverSampler(random_state=42)
    metricas = {'accuracy': [], 'precision_macro': [], 'recall_macro': [], 'f1_macro': [],
                'precision_pos': [], 'recall_pos': [], 'f1_pos': [], 'auc': []}

    print(f'\n{"="*70}')
    print(f'{nombre_modelo} — Experiment B — corrected macro-F1')
    print(f'{"="*70}')

    t_total = time.time()
    for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y), 1):
        t0 = time.time()
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        X_tr_b, y_tr_b = ros.fit_resample(X_tr, y_tr)

        modelo.fit(X_tr_b, y_tr_b)
        yp = modelo.predict(X_te)
        yp_pr = modelo.predict_proba(X_te)[:, 1]

        acc = accuracy_score(y_te, yp)
        pre_m = precision_score(y_te, yp, average='macro', zero_division=0)
        rec_m = recall_score(y_te, yp, average='macro', zero_division=0)
        f1_m  = f1_score(y_te, yp, average='macro', zero_division=0)
        pre_p = precision_score(y_te, yp, zero_division=0)  # positive-class only (paper's current definition)
        rec_p = recall_score(y_te, yp, zero_division=0)
        f1_p  = f1_score(y_te, yp, zero_division=0)
        auc = roc_auc_score(y_te, yp_pr)
        elapsed = time.time() - t0

        for m, v in zip(['accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
                          'precision_pos', 'recall_pos', 'f1_pos', 'auc'],
                         [acc, pre_m, rec_m, f1_m, pre_p, rec_p, f1_p, auc]):
            metricas[m].append(v)

        print(f'  Fold {fold:<3} macro-F1={f1_m:.4f}  pos-class-F1={f1_p:.4f}  '
              f'AUC={auc:.4f}  ({elapsed:.1f}s)')

    t_total = time.time() - t_total
    print('-' * 66)
    for m in metricas:
        print(f'  Mean {m:<16} {np.mean(metricas[m]):.4f} ± {np.std(metricas[m]):.4f}')
    print(f'  Total time: {t_total/60:.1f} min')

    return dict(metricas=metricas,
                media={m: np.mean(v) for m, v in metricas.items()},
                std={m: np.std(v) for m, v in metricas.items()})


# ── Random Forest, using the value the ORIGINAL RandomizedSearchCV actually
#    found for Experiment B (criterion='entropy'), not the 'gini' the current
#    pipeline uses by mistake (Comment 7) ──────────────────────────────────
rf_B_corrected = RandomForestClassifier(
    criterion='entropy', max_features='sqrt', bootstrap=True,
    max_samples=2/3, n_estimators=100, n_jobs=-1, random_state=42
)
res_rf_B_macro = validacion_cruzada_binaria_macro(X_B2, y_B2, rf_B_corrected, 'Random Forest (entropy, corrected)')

# ── Gradient Boosting: hyperparameters unchanged (max_features='sqrt' was
#    already what the original search found), only the averaging is fixed ──
gb_B_corrected = GradientBoostingClassifier(
    max_features='sqrt', learning_rate=0.1, n_estimators=100, random_state=42
)
res_gb_B_macro = validacion_cruzada_binaria_macro(X_B2, y_B2, gb_B_corrected, 'Gradient Boosting (corrected)')

print(f"\n{'='*70}")
print('SUMMARY — what changes in the manuscript')
print(f"{'='*70}")
print(f"Currently published (mislabeled as F1-macro): RF={0.7706}, GB={0.7242}")
print(f"Corrected positive-class F1 (RF, entropy):     {res_rf_B_macro['media']['f1_pos']:.4f}")
print(f"Corrected TRUE macro-F1 (RF, entropy):          {res_rf_B_macro['media']['f1_macro']:.4f}")
print(f"Corrected positive-class F1 (GB):               {res_gb_B_macro['media']['f1_pos']:.4f}")
print(f"Corrected TRUE macro-F1 (GB):                   {res_gb_B_macro['media']['f1_macro']:.4f}")



Random Forest (entropy, corrected) — Experiment B — corrected macro-F1
  Fold 1   macro-F1=0.5852  pos-class-F1=0.7701  AUC=0.7072  (17.4s)
  Fold 2   macro-F1=0.5876  pos-class-F1=0.7732  AUC=0.7135  (11.5s)
  Fold 3   macro-F1=0.5824  pos-class-F1=0.7661  AUC=0.7061  (12.2s)
  Fold 4   macro-F1=0.5917  pos-class-F1=0.7748  AUC=0.7160  (12.5s)
  Fold 5   macro-F1=0.5823  pos-class-F1=0.7716  AUC=0.6987  (14.4s)
  Fold 6   macro-F1=0.5848  pos-class-F1=0.7693  AUC=0.7081  (11.6s)
  Fold 7   macro-F1=0.5887  pos-class-F1=0.7726  AUC=0.7146  (13.4s)
  Fold 8   macro-F1=0.5864  pos-class-F1=0.7685  AUC=0.7130  (13.9s)
  Fold 9   macro-F1=0.5861  pos-class-F1=0.7690  AUC=0.7092  (14.3s)
  Fold 10  macro-F1=0.5877  pos-class-F1=0.7701  AUC=0.7134  (13.4s)
------------------------------------------------------------------
  Mean accuracy         0.6683 ± 0.0028
  Mean precision_macro  0.5935 ± 0.0024
  Mean recall_macro     0.6468 ± 0.0039
  Mean f1_macro         0.5863 ± 0.0027
  Mean prec